# MorphoNup — all datasets

Status of every dataset × k, and the cross-dataset benchmark tables and figures (refit, mean ± SD over the finished seeds). Training is started from `01_Train_MorphoNup.ipynb` or `scripts/run_cv.py`; this notebook only reads results.

In [ ]:
K_LIST = [5, 10]
SEEDS = [151000, 152000, 153000]

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
_null, _err = os.open(os.devnull, os.O_WRONLY), os.dup(2)
os.dup2(_null, 2)                                    # TensorFlow's CUDA start-up lines go to C-level stderr
try:
    import tensorflow  # noqa: F401
finally:
    os.dup2(_err, 2)
    os.close(_null)
    os.close(_err)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "morphonup" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display

from morphonup import config, figures, jobs
from morphonup.cv import run_dir

pd.set_option("display.width", 200, "display.max_columns", 40)


def show(path, width=900):
    """Display a saved figure with its caption (from the folder's figure_catalog.csv)."""
    path = Path(path)
    if not path.exists():
        print("missing:", path.relative_to(ROOT))
        return
    cat = path.parent / "figure_catalog.csv"
    caption = pd.read_csv(cat).set_index("png").suggested_caption.get(path.name, "") if cat.exists() else ""
    display(Markdown(f"**{path.stem}** — `{path.relative_to(ROOT)}`  \n{caption}"))
    display(Image(filename=str(path), width=width))

gpu, procs = jobs.gpu_status()
print("GPU:", gpu, "| processes:", procs or "none")
print("MorphoNup jobs running:", jobs.running_jobs() or "none")

## Status

In [ ]:
status = pd.concat([jobs.progress(d, k, SEEDS) for d in config.DATASETS for k in K_LIST], ignore_index=True)
overview = (status.assign(done=status.status.eq("done"), started=status.status.eq("started"))
            .groupby(["dataset", "k"], sort=False)[["done", "started"]].sum().unstack("k"))
display(overview)

## Benchmark

In [ ]:
drawn = figures.benchmark_figures(config.RESULTS_ROOT)
table = config.RESULTS_ROOT / "benchmark" / "benchmark_table.md"
display(Markdown(table.read_text()) if table.exists() else Markdown("_No finished dataset yet._"))

In [ ]:
BENCH = config.RESULTS_ROOT / "benchmark" / "figures"
for k in K_LIST:
    if (BENCH / f"heatmap_k{k}.png").exists():
        show(BENCH / f"heatmap_k{k}.png")
    else:
        print(f"k={k}: no finished dataset yet")
if (BENCH / "k_comparison.png").exists():
    show(BENCH / "k_comparison.png")
else:
    print("k comparison: needs a dataset finished at both k")